<a href="https://colab.research.google.com/github/Surajsurya95096/My-Bot-Deployer/blob/main/deploy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title 🚀 **Universal Bot Deployer** { display-mode: "form" }

# @markdown ### ⚙️ Heroku Setup
Heroku_Email = ""  # @param {type:"string"}
Heroku_API_Key = ""  # @param {type:"string"}
Heroku_App_Name = ""  # @param {type:"string"}

# @markdown ---
# @markdown ### 🔒 GitHub Settings
Git_Repo_URL = ""  # @param {type:"string"}
Git_Branch = ""  # @param {type:"string"}
GitHub_Access_Token = ""  # @param {type:"string"}

# @markdown ---
# @markdown ### 🤖 Core Variables (har bot ke liye common)
API_ID = ""  # @param {type:"string"}
API_HASH = ""  # @param {type:"string"}
BOT_TOKEN = ""  # @param {type:"string"}
MONGO_URI = ""  # @param {type:"string"}
OWNER_ID = ""  # @param {type:"string"}

# @markdown ---
# @markdown ### ➕ Extra Variables (optional — is bot ko koi aur env var chahiye to yahan daalo)
# @markdown Ek line mein ek variable, format: `KEY=VALUE`. Khali chhod sakte ho.
Extra_Env_Vars = ""  # @param {type:"string"}

import os
import re
import subprocess
import sys
import time

os.chdir("/content")

GREEN = "\033[92m"
RED = "\033[91m"
YELLOW = "\033[93m"
RESET = "\033[0m"

def run_cmd(cmd, check=True, cwd="/content"):
    res = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd=cwd)
    if check and res.returncode != 0:
        print(f"{RED}Error: {res.stderr.strip()}{RESET}")
    return res.returncode, res.stdout, res.stderr

# ---------------------------------------------------------------------
# Validation — only the 5 core fields (+ Heroku/GitHub setup) are ever
# required. Any bot-specific extras go through Extra_Env_Vars instead of
# needing this notebook edited per-bot.
# ---------------------------------------------------------------------
if not Heroku_Email or not Heroku_API_Key or not Heroku_App_Name or not Git_Repo_URL:
    print(f"{RED}[!] Heroku Email, API Key, App Name aur Repo URL sabhi bharein!{RESET}")
    sys.exit(1)

if not API_ID or not API_HASH or not BOT_TOKEN or not MONGO_URI or not OWNER_ID:
    print(f"{RED}[!] Saare 5 core variables (API_ID, API_HASH, BOT_TOKEN, MONGO_URI, OWNER_ID) bharein!{RESET}")
    sys.exit(1)

target_branch = Git_Branch.strip() if Git_Branch.strip() else "master"

# ---------------------------------------------------------------------
# Core env vars every bot deployed with this notebook gets. MONGO_URI is
# set under BOTH names since different bot codebases read this from
# either var name — harmless to set both, and means this notebook works
# regardless of which convention a given bot's config.py follows.
# ---------------------------------------------------------------------
env_dict = {
    "API_ID": API_ID.strip(),
    "API_HASH": API_HASH.strip(),
    "BOT_TOKEN": BOT_TOKEN.strip(),
    "MONGO_URI": MONGO_URI.strip(),
    "DATABASE_URL": MONGO_URI.strip(),
    "OWNER_ID": OWNER_ID.strip(),
    "WEB_CONCURRENCY": "1",
}

# ---------------------------------------------------------------------
# Parse free-form Extra_Env_Vars (KEY=VALUE per line). Any of these can
# override a core var above if the bot needs something different (e.g.
# a bot that only reads DATABASE_URL and would be confused by MONGO_URI
# also being set — just leave it, both being present is harmless for
# almost every bot, but this override path exists just in case).
# ---------------------------------------------------------------------
extra_count = 0
for line in Extra_Env_Vars.splitlines():
    line = line.strip()
    if not line or line.startswith("#") or "=" not in line:
        continue
    key, _, value = line.partition("=")
    key = key.strip()
    value = value.strip()
    if key:
        env_dict[key] = value
        extra_count += 1

formatted_repo_url = Git_Repo_URL.strip()
if GitHub_Access_Token.strip():
    clean_url = formatted_repo_url.replace("https://", "").replace("http://", "").split("@")[-1]
    formatted_repo_url = f"https://{GitHub_Access_Token.strip()}@{clean_url}"

print(f"{YELLOW}[1/6] Heroku CLI verify ho rahi hai...{RESET}")
subprocess.run("curl -s https://cli-assets.heroku.com/install.sh | sh > /dev/null 2>&1", shell=True, cwd="/content")

print(f"{YELLOW}[2/6] Heroku login config ho raha hai...{RESET}")
netrc_data = f"machine api.heroku.com\n  login {Heroku_Email.strip()}\n  password {Heroku_API_Key.strip()}\nmachine git.heroku.com\n  login {Heroku_Email.strip()}\n  password {Heroku_API_Key.strip()}\n"
with open(os.path.expanduser("~/.netrc"), "w") as f:
    f.write(netrc_data)
os.chmod(os.path.expanduser("~/.netrc"), 0o600)

app_name = Heroku_App_Name.strip().lower()
print(f"{YELLOW}[3/6] App check ki ja rahi hai...{RESET}")
create_code, out, err = run_cmd(f"heroku create {app_name}", check=False)
if create_code != 0:
    info_code, _, _ = run_cmd(f"heroku apps:info -a {app_name}", check=False)
    if info_code != 0:
        print(f"{RED}[!] App Name galat ya already taken hai!{RESET}")
        sys.exit(1)

# ---------------------------------------------------------------------
# Buildpacks: apt + python, unconditionally. If a bot repo has no
# Aptfile, the apt buildpack simply installs nothing and passes through
# harmlessly — so it's always safe to add both regardless of what a
# given bot actually needs, and this notebook never has to know.
# ---------------------------------------------------------------------
print(f"{YELLOW}[3.5/6] Buildpacks set ho rahe hain (apt + python)...{RESET}")
run_cmd(f"heroku buildpacks:clear -a {app_name}", check=False)
run_cmd(f"heroku buildpacks:add --index 1 heroku-community/apt -a {app_name}", check=False)
bp_code, _, bp_err = run_cmd(f"heroku buildpacks:add --index 2 heroku/python -a {app_name}", check=False)
if bp_code == 0:
    print(f"{GREEN}✅ Buildpacks set: apt + python{RESET}")
else:
    print(f"{RED}[!] Buildpack set karne mein error: {bp_err.strip()}{RESET}")

print(f"{YELLOW}[4/6] Repo clone ho raha hai ({target_branch})...{RESET}")
repo_dir = "/content/bot_deploy"
if os.path.exists(repo_dir):
    run_cmd(f"rm -rf {repo_dir}")

code, _, err = run_cmd(f"git clone -b {target_branch} {formatted_repo_url} {repo_dir}")
if code != 0:
    print(f"{RED}[❌] Git Clone fail ho gaya! Branch '{target_branch}' ya token check karein.{RESET}")
    sys.exit(1)

os.chdir(repo_dir)

run_cmd('git config --global user.email "deployer@colab.local"')
run_cmd('git config --global user.name "Colab Deployer"')

# ---------------------------------------------------------------------
# Auto-detect: does THIS bot repo need Playwright/Chromium? Only true if
# requirements.txt actually lists playwright — so bots that don't need
# it never pay the ~170MB download / longer build time, and this stays
# generic across any bot repo pointed at this notebook.
# ---------------------------------------------------------------------
needs_playwright = False
req_path = os.path.join(repo_dir, "requirements.txt")
if os.path.exists(req_path):
    with open(req_path, "r", errors="ignore") as f:
        req_text = f.read().lower()
    needs_playwright = bool(re.search(r"^playwright\b", req_text, re.MULTILINE))

if needs_playwright:
    print(f"{YELLOW}[*] Playwright detected in requirements.txt — Chromium build hook set ho raha hai...{RESET}")

    # PLAYWRIGHT_BROWSERS_PATH=0 is required so the browser installs
    # alongside the Python package in site-packages (part of the slug)
    # instead of /app/.cache (which Heroku does NOT carry from build to
    # the running dyno — verified directly: the build log shows a
    # successful download, but the runtime dyno reports the file missing
    # without this var set).
    env_dict["PLAYWRIGHT_BROWSERS_PATH"] = "0"

    os.makedirs(os.path.join(repo_dir, "bin"), exist_ok=True)
    post_compile_path = os.path.join(repo_dir, "bin", "post_compile")
    if not os.path.exists(post_compile_path):
        with open(post_compile_path, "w") as f:
            f.write(
                "#!/usr/bin/env bash\n"
                "set -e\n"
                "echo '-----> Installing Playwright Chromium browser'\n"
                "export PLAYWRIGHT_BROWSERS_PATH=0\n"
                "playwright install chromium 2>/dev/null || true\n"
            )
        os.chmod(post_compile_path, 0o755)
        # NOTE: git add/commit run with an explicit cwd=repo_dir rather
        # than through run_cmd() (which is hardcoded to cwd="/content")
        # — and add alone only stages the file, so an explicit commit is
        # required for it to actually reach what gets pushed below. Both
        # were real bugs found via testing on an earlier version of this
        # notebook.
        subprocess.run("git add bin/post_compile", shell=True, cwd=repo_dir, capture_output=True, text=True)
        commit_res = subprocess.run(
            'git commit -m "Add Playwright Chromium build hook (bin/post_compile)"',
            shell=True, cwd=repo_dir, capture_output=True, text=True,
        )
        if commit_res.returncode == 0:
            print(f"{GREEN}✅ bin/post_compile committed{RESET}")
        else:
            print(f"{YELLOW}[!] bin/post_compile commit skipped or failed: {commit_res.stderr.strip()}{RESET}")
    else:
        print(f"{GREEN}✅ bin/post_compile already present in repo{RESET}")
else:
    print(f"{YELLOW}[*] Playwright not detected — skipping Chromium build hook.{RESET}")

print(f"{YELLOW}[*] Variables Heroku par set ho rahe hain...{RESET}")
config_args = [f'{k}="{v}"' for k, v in env_dict.items()]
set_code, _, set_err = run_cmd(f"heroku config:set {' '.join(config_args)} -a {app_name}", check=False)
if set_code == 0:
    extra_note = f" (+{extra_count} extra)" if extra_count else ""
    print(f"{GREEN}✅ {len(env_dict)} variables set ho gaye!{extra_note}{RESET}")
else:
    print(f"{RED}Config error: {set_err.strip()}{RESET}")

print(f"{GREEN}[5/6] Heroku par push ho raha hai...{RESET}\n")
deploy = subprocess.Popen(f"git push https://git.heroku.com/{app_name}.git HEAD:main --force", shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in deploy.stdout:
    print(line, end="")
deploy.wait()

if deploy.returncode == 0:
    print(f"\n{GREEN}✅ Deployed Successfully!{RESET}")

    # -------------------------------------------------------------
    # Auto-detect dyno type from the repo's own Procfile instead of
    # assuming every bot is a worker — a bot whose Procfile declares
    # `web: ...` needs `web=1`, not `worker=1`, or nothing actually
    # starts. Falls back to worker if no Procfile is found (most
    # Telegram bots are workers) so this never leaves a bot unscaled.
    # -------------------------------------------------------------
    print(f"{YELLOW}[6/6] Dyno type Procfile se detect ho raha hai...{RESET}")
    procfile_path = os.path.join(repo_dir, "Procfile")
    process_types = []
    if os.path.exists(procfile_path):
        with open(procfile_path, "r", errors="ignore") as f:
            for line in f:
                line = line.strip()
                if line and ":" in line and not line.startswith("#"):
                    process_types.append(line.split(":", 1)[0].strip())

    if not process_types:
        process_types = ["worker"]
        print(f"{YELLOW}[!] Procfile nahi mila ya khali tha — default 'worker' use kar rahe hain.{RESET}")

    # Scale every process type this Procfile actually declares to 1.
    #
    # IMPORTANT: we do NOT blanket-zero out "the other" of {worker, web}
    # here. `heroku ps:scale web=0` errors with "Couldn't find that
    # process type" if `web` has never existed on this app (Heroku only
    # knows about process types a Procfile has declared at least once) —
    # this was a real bug found via testing: it made the WHOLE scale
    # command fail (including the worker=1 half), so the bot never
    # actually started despite a successful build. Instead, we only
    # zero out a previously-existing OTHER process type if `heroku ps`
    # shows it's actually present on this app already (e.g. a redeploy
    # that switched a bot from web to worker) — never touch a process
    # type that has no history on this app at all.
    scale_targets = {pt: 1 for pt in process_types}
    _, ps_out, _ = run_cmd(f"heroku ps -a {app_name}", check=False)
    for other in ("worker", "web"):
        if other not in scale_targets and re.search(rf"^{other}\b", ps_out, re.MULTILINE):
            scale_targets[other] = 0
    scale_args = " ".join(f"{k}={v}" for k, v in scale_targets.items())
    print(f"{YELLOW}[*] Scaling: {scale_args}{RESET}")
    run_cmd(f"heroku ps:scale {scale_args} -a {app_name}")

    time.sleep(2)
    log_proc = subprocess.Popen(f"heroku logs --tail -a {app_name}", shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    try:
        for log_line in log_proc.stdout:
            print(log_line, end="")
    except KeyboardInterrupt:
        pass
else:
    print(f"\n{RED}❌ Build fail hui.{RESET}")

os.chdir("/content")